In [1]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = os.getenv("CLAUDE_MODEL")

In [2]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [3]:
import json

# For step 2, where we create an evaluation dataset. We ask Claude to do it for us.
def generate_dataset():
    prompt = """
		Generate a evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts
		that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects,
		each representing task that requires Python, JSON, or a Regex to complete.

		Example output:
		```json
		[
			{
				"task": "Description of task",
			},
			...additional
		]
		```

		* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a regular expression.
		* Focus on tasks that do not require writing much code

		Please generate 3 objects.
	"""
    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences=["```"])
    return json.loads(text)

In [5]:
dataset = generate_dataset()

with open("dataset.json", "w") as file:
  json.dump(dataset, file, indent=2)

dataset

[{'task': "Write a Python function that extracts the AWS region from an S3 bucket URI in the format 's3://bucket-name/key'. If no region is specified, return 'us-east-1' as default."},
 {'task': "Create a JSON object representing an AWS IAM policy that grants read-only access to all objects in an S3 bucket named 'my-data-bucket'."},
 {'task': "Write a regular expression that matches valid AWS EC2 instance IDs, which follow the pattern of 'i-' followed by 8 or 17 hexadecimal characters."}]